### **Chapter 9.3: DeePC and MPC on a Deterministic LTI System**

For deterministic controllable LTI systems with sufficiently rich data, DeePC and model-based MPC optimize over the same set of finite trajectories. This notebook demonstrates that equivalence numerically on the **flat Mountain Car**.

The comparison is deliberately controlled:

- same plant and sampling time;
- same $Q$, $R$, $Q_f$ and prediction horizon $N$;
- same input constraints and reference;
- DeePC uses only the offline Hankel data;
- MPC uses the explicit linear model from Chapter 5.

For this theorem-oriented experiment we set the DeePC regularization essentially to zero.

In [ ]:
import sys
import os
import time
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from utils.env import *
from utils.controller import *
from utils.simulator import *
from ex5_MPC.mpc_utils import LinearMPCController
from ex9_DeePC.deepc_utils import *

### **Part 1: Identical Control Problem**

In [ ]:
case = 1
freq = 20
t_terminal = 8
N = 20
T_ini = 4

initial_state = np.array([-0.5, 0.0])
target_state = np.array([0.6, 0.0])

env = Env(
    case,
    initial_state,
    target_state,
    input_lbs=-1.0,
    input_ubs=1.0,
)
dynamics = Dynamics(env)

Q = np.diag([1.0, 1.0])
R = np.array([[0.1]])
Qf = Q

### **Part 2: Build DeePC from Data**

No $A$ or $B$ matrix is passed to the DeePC controller. Its predictive behavior comes from the offline trajectory.

In [ ]:
u_data, y_data = collect_deepc_data(
    env,
    dynamics,
    freq=freq,
    n_samples=400,
    excitation_amplitude=0.8,
    initial_state=env.target_state,
    seed=1,
)

controller_deepc = DeePCController(
    env,
    dynamics,
    u_data,
    y_data,
    Q,
    R,
    Qf,
    freq,
    N,
    T_ini=T_ini,
    lambda_g=1e-10,  # only numerical regularization for the equivalence demo
    history_initialization='equilibrium',
    name='DeePC_equivalence',
    verbose=False,
)

### **Part 3: Build the Chapter-5 Linear MPC**

The model-based controller uses the linear dynamics explicitly. Apart from the trajectory representation, the online optimization objective and constraints are matched to DeePC.

In [ ]:
controller_mpc = LinearMPCController(
    env,
    dynamics,
    Q,
    R,
    Qf,
    freq,
    N,
    name='Linear_MPC',
    verbose=False,
)

### **Part 4: Closed-Loop Comparison**

In [ ]:
start_time = time.time()
simulator_deepc = Simulator(dynamics, controller_deepc, env, 1/freq, t_terminal)
simulator_deepc.run_simulation()
time_deepc = time.time() - start_time

start_time = time.time()
simulator_mpc = Simulator(dynamics, controller_mpc, env, 1/freq, t_terminal)
simulator_mpc.run_simulation()
time_mpc = time.time() - start_time

x_deepc, u_deepc = simulator_deepc.get_trajectories()
x_mpc, u_mpc = simulator_mpc.get_trajectories()

u_deepc = np.asarray(u_deepc).reshape(-1, dynamics.dim_inputs)
u_mpc = np.asarray(u_mpc).reshape(-1, dynamics.dim_inputs)

print(f"Max state difference: {np.max(np.linalg.norm(x_deepc - x_mpc, axis=1)):.3e}")
print(f"Max input difference: {np.max(np.linalg.norm(u_deepc - u_mpc, axis=1)):.3e}")
print(f"Mean DeePC solve+simulation time per step: {time_deepc/(freq*t_terminal):.6f} s")
print(f"Mean MPC solve+simulation time per step:   {time_mpc/(freq*t_terminal):.6f} s")

In [ ]:
t_x = np.arange(x_deepc.shape[0]) / freq
t_u = np.arange(u_deepc.shape[0]) / freq

fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
ax[0].plot(t_x, x_mpc[:, 0], label="MPC")
ax[0].plot(t_x, x_deepc[:, 0], "--", label="DeePC")
ax[0].axhline(env.target_position, linestyle=":", label="target")
ax[0].set_ylabel("position")
ax[0].legend()

ax[1].plot(t_x, x_mpc[:, 1], label="MPC")
ax[1].plot(t_x, x_deepc[:, 1], "--", label="DeePC")
ax[1].set_ylabel("velocity")

ax[2].plot(t_u, u_mpc[:, 0], label="MPC")
ax[2].plot(t_u, u_deepc[:, 0], "--", label="DeePC")
ax[2].set_ylabel("input")
ax[2].set_xlabel("Time (s)")

fig.suptitle("Deterministic LTI closed-loop behavior")
plt.tight_layout()
plt.show()

### **Part 5: Error and Cost Comparison**

If both formulations optimize over the same finite behavior, their differences should be dominated by numerical solver/integration tolerances.

In [ ]:
state_error = np.linalg.norm(x_deepc - x_mpc, axis=1)
input_error = np.linalg.norm(u_deepc - u_mpc, axis=1)

def quadratic_closed_loop_cost(x, u, x_ref, u_ref, Q, R, Qf):
    dx = x - x_ref
    du = u - u_ref
    J = sum(dx[k] @ Q @ dx[k] + du[k] @ R @ du[k] for k in range(len(u)))
    J += dx[-1] @ Qf @ dx[-1]
    return float(J)

u_ref = np.atleast_1d(dynamics.get_equilibrium_input(env.target_state))
J_deepc = quadratic_closed_loop_cost(x_deepc, u_deepc, env.target_state, u_ref, Q, R, Qf)
J_mpc = quadratic_closed_loop_cost(x_mpc, u_mpc, env.target_state, u_ref, Q, R, Qf)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].semilogy(t_x, np.maximum(state_error, 1e-16))
ax[0].set_xlabel("Time (s)")
ax[0].set_ylabel(r"$\|x^{DeePC}-x^{MPC}\|_2$")
ax[0].set_title("State difference")

ax[1].semilogy(t_u, np.maximum(input_error, 1e-16))
ax[1].set_xlabel("Time (s)")
ax[1].set_ylabel(r"$\|u^{DeePC}-u^{MPC}\|_2$")
ax[1].set_title("Input difference")
plt.tight_layout()
plt.show()

print(f"Closed-loop cost DeePC: {J_deepc:.8f}")
print(f"Closed-loop cost MPC:   {J_mpc:.8f}")
print(f"Cost difference:        {abs(J_deepc-J_mpc):.3e}")

<blockquote style="padding: 18px 20px; margin: 1.2em 0; background: rgba(56, 139, 253, 0.12); border-left: 4px solid rgba(56, 139, 253, 0.85); border-radius: 6px; color: inherit !important;">

##### **Takeaway: DeePC replaces the model-based trajectory generator, not the predictive-control idea**

For deterministic LTI systems, sufficiently rich Hankel data and an explicit $(A,B)$ model describe the same finite behavior. The objective, constraints, receding-horizon loop, and first-input implementation can therefore be the same.
</blockquote>

The constant-slope example in Chapter 9.1 is also linear after equilibrium shifting. For the clean numerical equivalence figure here we use the flat case because the Chapter-5 `LinearMPCController` is written directly for homogeneous linear dynamics and does not explicitly include an affine offset.

**Reference:** Coulson, Lygeros, and Dörfler, *Data-Enabled Predictive Control: In the Shallows of the DeePC*, Theorem 5.1 and Corollary 5.1.